<a href="https://colab.research.google.com/github/zm-practice/LCPU/blob/main/Notebooks/ATE_Training_%26_Inference.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Libraries

In [1]:
# 修复 utils.py 里的 tokenizer 参数
utils_path = '/content/drive/MyDrive/InstructABSA-main/InstructABSA/utils.py'

with open(utils_path, 'r') as f:
    content = f.read()

# 把 tokenizer= 改成 processing_class=
content = content.replace('tokenizer=self.tokenizer', 'processing_class=self.tokenizer')

with open(utils_path, 'w') as f:
    f.write(content)

print("修复完成")

修复完成


In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os
# 列出你Drive里的文件夹
print(os.listdir('/content/drive/MyDrive'))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
['校赛A.docx', 'Google AI Studio', 'Resume.gdoc', 'SemEval14', 'InstructABSA-main']


In [3]:
try:
    import google.colab
    from google.colab import drive
    drive.mount('/content/drive', force_remount = True)
    IN_COLAB = True
except:
    IN_COLAB = False

Mounted at /content/drive


In [4]:
if IN_COLAB:
  !pip install transformers
  !pip install datasets
  !pip install evaluate
  !pip install sentencepiece

In [5]:
import os
import torch

if IN_COLAB:
    root_path = '/content/drive/MyDrive/InstructABSA-main'
else:
    root_path = 'Enter local path'

use_mps = True if torch.has_mps else False
os.chdir(root_path)

/tmp/ipykernel_19539/2725293108.py:9: UserWarning: 'has_mps' is deprecated, please use 'torch.backends.mps.is_built()'
  use_mps = True if torch.has_mps else False


In [6]:
import warnings
warnings.filterwarnings('ignore')
import pandas as pd

from InstructABSA.data_prep import DatasetLoader
from InstructABSA.utils import T5Generator, T5Classifier
from instructions import InstructionsHandler

## Training

In [7]:
task_name = 'ate'
#experiment_name = 'lapt2014_iabsa1'
experiment_name = 'lapt2014_iabsa2'
#model_checkpoint = 'allenai/tk-instruct-base-def-pos'
model_checkpoint = 'allenai/tk-instruct-base-def-pos'
print('Experiment Name: ', experiment_name)
model_out_path = './Models'
model_out_path = os.path.join(model_out_path, task_name, f"{model_checkpoint.replace('/', '')}-{experiment_name}")
print('Model output path: ', model_out_path)

Experiment Name:  lapt2014_iabsa2
Model output path:  ./Models/ate/allenaitk-instruct-base-def-pos-lapt2014_iabsa2


In [8]:
# Load the data
id_train_file_path = './Dataset/SemEval14/Train/Laptops_Train.csv'
id_test_file_path = './Dataset/SemEval14/Test/Laptops_Test.csv'
id_tr_df = pd.read_csv(id_train_file_path)
id_te_df = pd.read_csv(id_test_file_path)

# Get the input text into the required format using Instructions
instruct_handler = InstructionsHandler()

# Set instruction_set1 for InstructABSA-1 and instruction_set2 for InstructABSA-2
instruct_handler.load_instruction_set2()

# Set bos_instruct1 for lapt14 and bos_instruct2 for rest14. For other datasets, modify the insructions.py file.
loader = DatasetLoader(id_tr_df, id_te_df)
if loader.train_df_id is not None:
    loader.train_df_id = loader.create_data_in_ate_format(loader.train_df_id, 'term', 'raw_text', 'aspectTerms', instruct_handler.ate['bos_instruct1'], instruct_handler.ate['eos_instruct'])
if loader.test_df_id is not None:
    loader.test_df_id = loader.create_data_in_ate_format(loader.test_df_id, 'term', 'raw_text', 'aspectTerms', instruct_handler.ate['bos_instruct1'], instruct_handler.ate['eos_instruct'])

In [9]:
# Create T5 utils object
t5_exp = T5Generator(model_checkpoint)

# Tokenize Dataset
id_ds, id_tokenized_ds, ood_ds, ood_tokenized_ds = loader.set_data_for_training_semeval(t5_exp.tokenize_function_inputs)

# Training arguments
training_args = {
    'output_dir':model_out_path,
    'eval_strategy':'no',#"epoch"没有验证集,#把 evaluation_strategy 改成 eval_strategy
    'learning_rate':5e-5,
    'lr_scheduler_type':'cosine',
    'per_device_train_batch_size':8,
    'per_device_eval_batch_size':16,
    'num_train_epochs':4,
    'weight_decay':0.01,
    'warmup_ratio':0.1,
    'save_strategy':'no',
    'load_best_model_at_end':False,
    'push_to_hub':False,
    'eval_accumulation_steps':1,
    'predict_with_generate':True,
    #'use_mps_device':use_mps
}

Loading weights:   0%|          | 0/284 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Map:   0%|          | 0/3045 [00:00<?, ? examples/s]

Map:   0%|          | 0/800 [00:00<?, ? examples/s]

In [11]:
# Train model
model_trainer = t5_exp.train(id_tokenized_ds, **training_args)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Trainer device: cuda:0

Model training started ....


Step,Training Loss
500,7.951937
1000,6.195625
1500,5.860563


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

## Inference

In [12]:
# Load the data
id_train_file_path = './Dataset/SemEval14/Train/Laptops_Train.csv'
id_test_file_path = './Dataset/SemEval14/Test/Laptops_Test.csv'
id_tr_df = pd.read_csv(id_train_file_path)
id_te_df = pd.read_csv(id_test_file_path)

# Get the input text into the required format using Instructions
instruct_handler = InstructionsHandler()

# Set instruction_set1 for InstructABSA-1 and instruction_set2 for InstructABSA-2
instruct_handler.load_instruction_set2()

# Set bos_instruct1 for lapt14 and bos_instruct2 for rest14. For other datasets, modify the insructions.py file.
loader = DatasetLoader(id_tr_df, id_te_df)
if loader.train_df_id is not None:
    loader.train_df_id = loader.create_data_in_ate_format(loader.train_df_id, 'term', 'raw_text', 'aspectTerms', instruct_handler.ate['bos_instruct1'], instruct_handler.ate['eos_instruct'])
if loader.test_df_id is not None:
    loader.test_df_id = loader.create_data_in_ate_format(loader.test_df_id, 'term', 'raw_text', 'aspectTerms', instruct_handler.ate['bos_instruct1'], instruct_handler.ate['eos_instruct'])

In [13]:
# Model inference - Loading from Checkpoint
t5_exp = T5Generator(model_out_path)

# Tokenize Datasets
id_ds, id_tokenized_ds, ood_ds, ood_tokenzed_ds = loader.set_data_for_training_semeval(t5_exp.tokenize_function_inputs)

# Get prediction labels - Training set
id_tr_pred_labels = t5_exp.get_labels(tokenized_dataset = id_tokenized_ds, sample_set = 'train',  batch_size = 16)
id_tr_labels = [i.strip() for i in id_ds['train']['labels']]
#不支持trained_model_path = model_out_path,参数
# Get prediction labels - Testing set
id_te_pred_labels = t5_exp.get_labels(tokenized_dataset = id_tokenized_ds, sample_set = 'test',  batch_size = 16)
id_te_labels = [i.strip() for i in id_ds['test']['labels']]

Loading weights:   0%|          | 0/284 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Map:   0%|          | 0/3045 [00:00<?, ? examples/s]

Map:   0%|          | 0/800 [00:00<?, ? examples/s]

Model loaded to:  cuda


100%|██████████| 191/191 [01:47<00:00,  1.78it/s]


Model loaded to:  cuda


100%|██████████| 50/50 [00:27<00:00,  1.79it/s]


In [14]:
p, r, f1 ,_= t5_exp.get_metrics(id_tr_labels, id_tr_pred_labels)#少了一个参数，加入_
print('Train Precision: ', p)
print('Train Recall: ', r)
print('Train F1: ', f1)

p, r, f1,_ = t5_exp.get_metrics(id_te_labels, id_te_pred_labels)
print('Test Precision: ', p)
print('Test Recall: ', r)
print('Test F1: ', f1)

Train Precision:  0.5116584564860427
Train Recall:  0.3979565772669221
Train F1:  0.4477011494252874
Test Precision:  0.47375
Test Recall:  0.36724806201550386
Test F1:  0.4137554585152838


In [15]:
# 看前5条预测输出
for i, (gt, pred) in enumerate(zip(id_te_labels[:5], id_te_pred_labels[:5])):
    print(f"GT:   {gt}")
    print(f"PRED: {pred}")
    print()

GT:   Boot time
PRED: noaspectterm

GT:   tech support
PRED: noaspectterm

GT:   noaspectterm
PRED: noaspectterm

GT:   Set up
PRED: noaspectterm

GT:   Windows 8, touchscreen functions
PRED: noaspectterm



# 直接加载作者的模型，指令2



In [ ]:
# Load the data
id_train_file_path = './Dataset/SemEval14/Train/Laptops_Train.csv'
id_test_file_path = './Dataset/SemEval14/Test/Laptops_Test.csv'
id_tr_df = pd.read_csv(id_train_file_path)
id_te_df = pd.read_csv(id_test_file_path)

instruct_handler = InstructionsHandler()
instruct_handler.load_instruction_set2()  # 指令2

loader = DatasetLoader(id_tr_df, id_te_df)
if loader.train_df_id is not None:
    loader.train_df_id = loader.create_data_in_ate_format(
        loader.train_df_id, 'term', 'raw_text', 'aspectTerms',
        instruct_handler.ate['bos_instruct1'],
        instruct_handler.ate['eos_instruct'])
if loader.test_df_id is not None:
    loader.test_df_id = loader.create_data_in_ate_format(
        loader.test_df_id, 'term', 'raw_text', 'aspectTerms',
        instruct_handler.ate['bos_instruct1'],
        instruct_handler.ate['eos_instruct'])

In [ ]:
# 直接加载作者训练好的模型，不需要训练
model_out_path = 'kevinscaria/ate_tk-instruct-base-def-pos-neg-neut-laptops'
t5_exp = T5Generator(model_out_path)

id_ds, id_tokenized_ds, ood_ds, ood_tokenized_ds = loader.set_data_for_training_semeval(
    t5_exp.tokenize_function_inputs)

id_tr_pred_labels = t5_exp.get_labels(
    tokenized_dataset=id_tokenized_ds, sample_set='train', batch_size=16)
id_tr_labels = [i.strip() for i in id_ds['train']['labels']]

id_te_pred_labels = t5_exp.get_labels(
    tokenized_dataset=id_tokenized_ds, sample_set='test', batch_size=16)
id_te_labels = [i.strip() for i in id_ds['test']['labels']]

In [ ]:
p, r, f1, _ = t5_exp.get_metrics(id_tr_labels, id_tr_pred_labels)
print('Train Precision: ', p)
print('Train Recall: ', r)
print('Train F1: ', f1)

p, r, f1, _ = t5_exp.get_metrics(id_te_labels, id_te_pred_labels)
print('Test Precision: ', p)
print('Test Recall: ', r)
print('Test F1: ', f1)

# 直接加载作者的模型，指令1

In [ ]:
# Load the data
id_train_file_path = './Dataset/SemEval14/Train/Laptops_Train.csv'
id_test_file_path = './Dataset/SemEval14/Test/Laptops_Test.csv'
id_tr_df = pd.read_csv(id_train_file_path)
id_te_df = pd.read_csv(id_test_file_path)

instruct_handler = InstructionsHandler()
instruct_handler.load_instruction_set1()  # 指令1

loader = DatasetLoader(id_tr_df, id_te_df)
if loader.train_df_id is not None:
    loader.train_df_id = loader.create_data_in_ate_format(
        loader.train_df_id, 'term', 'raw_text', 'aspectTerms',
        instruct_handler.ate['bos_instruct1'],
        instruct_handler.ate['eos_instruct'])
if loader.test_df_id is not None:
    loader.test_df_id = loader.create_data_in_ate_format(
        loader.test_df_id, 'term', 'raw_text', 'aspectTerms',
        instruct_handler.ate['bos_instruct1'],
        instruct_handler.ate['eos_instruct'])

In [ ]:
# 直接加载作者训练好的模型，不需要训练
model_out_path = 'kevinscaria/ate_tk-instruct-base-def-pos-laptops'
t5_exp = T5Generator(model_out_path)

id_ds, id_tokenized_ds, ood_ds, ood_tokenized_ds = loader.set_data_for_training_semeval(
    t5_exp.tokenize_function_inputs)

id_tr_pred_labels = t5_exp.get_labels(
    tokenized_dataset=id_tokenized_ds, sample_set='train', batch_size=16)
id_tr_labels = [i.strip() for i in id_ds['train']['labels']]

id_te_pred_labels = t5_exp.get_labels(
    tokenized_dataset=id_tokenized_ds, sample_set='test', batch_size=16)
id_te_labels = [i.strip() for i in id_ds['test']['labels']]

In [ ]:
p, r, f1, _ = t5_exp.get_metrics(id_tr_labels, id_tr_pred_labels)
print('Train Precision: ', p)
print('Train Recall: ', r)
print('Train F1: ', f1)

p, r, f1, _ = t5_exp.get_metrics(id_te_labels, id_te_pred_labels)
print('Test Precision: ', p)
print('Test Recall: ', r)
print('Test F1: ', f1)